In [2]:
from embedder import Embedder

embed = Embedder()

In [3]:
# Question 1: What is the first value of embedder
query = "How does approximate nearest neighbor search work?"
v1 = embed.encode(query)
v1[0]

np.float64(-0.02058203437252893)

#### Loading Data

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
# Question 2: Cosine Similarity
doc = next(doc for doc in documents if "07-sqlitesearch-vector.md" in doc["filename"])
v_doc = embed.encode(doc["content"])

similarity = v_doc.dot(v1)
print(f"Cosine similarity between the query and the document: {similarity:.4f}")

Cosine similarity between the query and the document: 0.3611


In [7]:
# Question 3: Chunking and Search by Hand
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
chunk_texts = [chunk["content"] for chunk in chunks]

X = embed.encode_batch(chunk_texts)

scores = X.dot(v1)

best_chunk_index = scores.argmax()

# find filename of the best chunk
best_chunk = chunks[best_chunk_index]["filename"]
print(f"Best chunk filename: {best_chunk} with score: {scores[best_chunk_index]:.4f}")


Best chunk filename: 02-vector-search/lessons/07-sqlitesearch-vector.md with score: 0.6489


In [15]:

from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

print(f"First result filename: {results[0]['filename']}")

First result filename: 04-evaluation/lessons/05-search-metrics.md


In [16]:
# Question 5: Text Search vs Vector Search
from minsearch import Index

query = "How do I store vectors in PostgreSQL?"

v = embed.encode(query)
v_results = vindex.search(v, num_results=5)
vector_filenames = [result["filename"] for result in v_results]

t_index = Index(text_fields=["content"], keyword_fields=["filename"])
t_index.fit(chunks)

text_results = t_index.search(query, num_results=5)
text_filenames = [result["filename"] for result in text_results]

difference = set(vector_filenames) - set(text_filenames)

print(f"Vector search results:")
for filename in vector_filenames:
    print(f"- {filename}")
    
print(f"\nText search results:")
for filename in text_filenames:
    print(f"- {filename}")
    
print(f"\nDifference (vector search results not in text search results):")
for filename in difference:
    print(f"- {filename}")

Vector search results:
- 02-vector-search/lessons/08-pgvector.md
- 02-vector-search/lessons/08-pgvector.md
- 03-orchestration/lessons/05-rag.md
- 02-vector-search/lessons/08-pgvector.md
- 02-vector-search/lessons/08-pgvector.md

Text search results:
- 02-vector-search/lessons/02-embeddings.md
- 03-orchestration/lessons/05-rag.md
- 02-vector-search/lessons/01-intro.md
- 03-orchestration/lessons/05-rag.md
- 02-vector-search/lessons/01-intro.md

Difference (vector search results not in text search results):
- 02-vector-search/lessons/08-pgvector.md


In [17]:
# Question 6: Hybrid Search
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [18]:
query = "How do I give the model access to tools?"

# Run the query with both vector and text search
v = embed.encode(query)
t_results = t_index.search(query, num_results=5)
v_results = vindex.search(v, num_results=5)
hybrid_results = rrf([t_results, v_results])

print(f"First result filename from hybrid search: {hybrid_results[0]['filename']}")

First result filename from hybrid search: 01-agentic-rag/lessons/13-function-calling.md
